In [2]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)   # empty slices where a whole row was dropped
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

DATA = '../data'
X_train = np.load(f'{DATA}/train_cadences.npy', mmap_mode='r')   # (18000, 6, 16, 128) uint8
X_test  = np.load(f'{DATA}/test_cadences.npy', mmap_mode='r')    # (10000, 6, 16, 128) uint8
train = pd.read_csv(f'{DATA}/train_meta.csv')
test  = pd.read_csv(f'{DATA}/test_meta.csv')
print(X_train.shape, X_test.shape)
print(train.label.value_counts().to_dict())
print("Training metadata columns:")
print(train.columns.tolist())

print("\nTraining metadata sample:")
display(train.head())

print("\nTraining metadata data types:")
print(train.dtypes)

print("\nValidation score:", 0.5461)

(18000, 6, 16, 128) (10000, 6, 16, 128)
{'noise': 4211, 'rfi_persistent': 2892, 'rfi_intermittent': 2662, 'radar_chirp': 1849, 'satellite_pass': 1805, 'pulsar': 1629, 'scintillating_star': 1431, 'frb': 1042, 'eti_candidate': 479}
Training metadata columns:
['id', 'telescope', 'target', 'center_freq_mhz', 'channel_width_hz', 'tsamp_s', 'slew_gap_rows', 'label']

Training metadata sample:


,id,telescope,target,center_freq_mhz,channel_width_hz,tsamp_s,slew_gap_rows,label
0,SH7cefe0e1,GBT,HIP59624,1384.518387,2.79,18.25,5,scintillating_star
1,SH1dbf97aa,GBT,HIP27609,1659.448768,2.79,18.25,6,satellite_pass
2,SHc19f6060,GBT,HIP45387,1627.507811,2.79,18.25,3,rfi_persistent
3,SH41a0663c,GBT,HIP67753,1107.781667,2.79,18.25,5,rfi_intermittent
4,SHdb4c65a1,GBT,HIP93142,1507.034261,2.79,18.25,4,pulsar



Training metadata data types:
id                      str
telescope               str
target                  str
center_freq_mhz     float64
channel_width_hz    float64
tsamp_s             float64
slew_gap_rows         int64
label                   str
dtype: object

Validation score: 0.5461


In [3]:
# E1 diagnostic: missingness, intensity distribution, and telescope metadata

print("Telescope distribution:")
print(pd.concat([
    train["telescope"].value_counts().rename("train"),
    test["telescope"].value_counts().rename("test")
], axis=1).fillna(0).astype(int))

print("\nSlew-gap distribution:")
print(train["slew_gap_rows"].describe())
print(test["slew_gap_rows"].describe())

def panel_diagnostics(X, name):
    print(f"\n{name}")
    for p in range(6):
        a = np.asarray(X[:, p], dtype=np.float32)
        valid = a != 0
        values = a[valid]

        print(
            f"Panel {p} ({'ON' if p % 2 == 0 else 'OFF'}): "
            f"missing={1 - valid.mean():.3%}, "
            f"valid_pixels={valid.sum():,}, "
            f"median={np.median(values) if values.size else np.nan:.2f}, "
            f"p95={np.percentile(values, 95) if values.size else np.nan:.2f}, "
            f"p99={np.percentile(values, 99) if values.size else np.nan:.2f}"
        )

panel_diagnostics(X_train, "TRAIN")
panel_diagnostics(X_test, "TEST")

Telescope distribution:
           train   test
telescope              
GBT        18000      0
PKS            0  10000

Slew-gap distribution:
count    18000.000000
mean         4.510333
std          1.121490
min          3.000000
25%          4.000000
50%          5.000000
75%          6.000000
max          6.000000
Name: slew_gap_rows, dtype: float64
count    10000.000000
mean         4.491200
std          1.117518
min          3.000000
25%          3.000000
50%          4.000000
75%          5.000000
max          6.000000
Name: slew_gap_rows, dtype: float64

TRAIN
Panel 0 (ON): missing=0.167%, valid_pixels=36,802,560, median=94.00, p95=118.00, p99=137.00
Panel 1 (OFF): missing=0.146%, valid_pixels=36,810,112, median=93.00, p95=116.00, p99=135.00
Panel 2 (ON): missing=0.152%, valid_pixels=36,807,936, median=94.00, p95=118.00, p99=137.00
Panel 3 (OFF): missing=0.147%, valid_pixels=36,809,984, median=93.00, p95=116.00, p99=135.00
Panel 4 (ON): missing=0.164%, valid_pixels=36,803,456, 

In [4]:
from scipy.ndimage import median_filter

def bandpass_features(X, batch=1000):
    """
    Extract complementary features after frequency-dependent
    bandpass correction. Zero pixels remain missing.
    """
    output = []

    for start in range(0, len(X), batch):
        raw = X[start:start + batch].astype(np.float32)
        raw[raw == 0] = np.nan

        # Remove each panel's overall level.
        centered = raw - np.nanmedian(
            raw, axis=(2, 3), keepdims=True
        )

        # Estimate the broad frequency profile using the median
        # over time, then smooth across frequency channels.
        profile = np.nanmedian(centered, axis=2)  # (B, 6, 128)

        # Fill missing profile entries temporarily for smoothing.
        # The correction is applied only to valid original pixels.
        profile_filled = np.where(np.isfinite(profile), profile, 0.0)

        smooth = median_filter(
            profile_filled,
            size=(1, 1, 9),
            mode="nearest"
        )

        corrected = centered - smooth[:, :, None, :]
        corrected[~np.isfinite(raw)] = np.nan

        feats = {}

        for p in range(6):
            panel = corrected[:, p]
            feats[f'bp_p{p}_std'] = np.nanstd(panel, axis=(1, 2))
            feats[f'bp_p{p}_max'] = np.nanmax(panel, axis=(1, 2))
            feats[f'bp_p{p}_peak'] = np.nanmax(
                np.nanmean(panel, axis=1), axis=1
            )
            feats[f'bp_p{p}_rowmax'] = np.nanmax(
                np.nanmean(panel, axis=2), axis=1
            )

        df = pd.DataFrame(feats)

        for k in ['std', 'max', 'peak', 'rowmax']:
            on = df[[f'bp_p{p}_{k}' for p in (0, 2, 4)]].mean(axis=1)
            off = df[[f'bp_p{p}_{k}' for p in (1, 3, 5)]].mean(axis=1)
            df[f'bp_on_minus_off_{k}'] = on - off

        output.append(df)

    return pd.concat(output, ignore_index=True)

In [6]:
def features(X, meta, batch=2000):
    out = []
    for s in range(0, len(X), batch):
        x = X[s:s + batch].astype(np.float32)                 # (b, 6, 16, 128)
        x[x == 0] = np.nan                                      # dropped rows
        x = x - np.nanmedian(x, axis=(2, 3), keepdims=True)     # remove the level of each panel
        prof = np.nanmean(x, axis=2)                            # (b, 6, 128): time-averaged spectrum
        f = {}
        for p in range(6):
            f[f'p{p}_std'] = np.nanstd(x[:, p], axis=(1, 2))
            f[f'p{p}_max'] = np.nanmax(x[:, p], axis=(1, 2))
            f[f'p{p}_peak'] = np.nanmax(prof[:, p], axis=1)
            f[f'p{p}_peak_ch'] = np.nanargmax(prof[:, p], axis=1)
            f[f'p{p}_rowmax'] = np.nanmax(np.nanmean(x[:, p], axis=2), axis=1)   # broadband events
        df = pd.DataFrame(f)
        for k in ['std', 'max', 'peak', 'rowmax']:
            on = df[[f'p{p}_{k}' for p in (0, 2, 4)]].mean(1)
            off = df[[f'p{p}_{k}' for p in (1, 3, 5)]].mean(1)
            df[f'on_minus_off_{k}'] = on - off
        out.append(df)
    F = pd.concat(out, ignore_index=True)
    return pd.concat([F, meta[['slew_gap_rows', 'tsamp_s', 'channel_width_hz']].reset_index(drop=True)], axis=1)

F_train = features(X_train, train)
F_test = features(X_test, test)
F_train.shape

C:\Users\kasir\AppData\Local\Temp\ipykernel_14460\250054591.py:17: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  on = df[[f'p{p}_{k}' for p in (0, 2, 4)]].mean(1)
C:\Users\kasir\AppData\Local\Temp\ipykernel_14460\250054591.py:18: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  off = df[[f'p{p}_{k}' for p in (1, 3, 5)]].mean(1)


(18000, 37)

In [11]:
F_bp_train = bandpass_features(X_train)
F_bp_test = bandpass_features(X_test)

# Retain the exact same validation rows and classifier settings.
F_combined = pd.concat(
    [F_train.reset_index(drop=True), F_bp_train],
    axis=1
)

X_tr, X_va, y_tr, y_va = train_test_split(
    F_combined,
    train.label,
    test_size=0.2,
    stratify=train.label,
    random_state=42
)

model_e1 = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.07,
    class_weight="balanced",
    random_state=0
)

model_e1.fit(X_tr, y_tr)
pred_e1 = model_e1.predict(X_va)

print("E1 macro-F1:", f1_score(y_va, pred_e1, average="macro"))

E1 macro-F1: 0.5721949757593479


In [12]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(
    y_va,
    pred_e1,
    digits=4,
    zero_division=0
))

                    precision    recall  f1-score   support

     eti_candidate     0.2000    0.0729    0.1069        96
               frb     0.7295    0.7225    0.7260       209
             noise     0.7169    0.8931    0.7953       842
            pulsar     0.8746    0.7485    0.8066       326
       radar_chirp     0.6937    0.7405    0.7163       370
  rfi_intermittent     0.5794    0.4117    0.4813       532
    rfi_persistent     0.6598    0.7785    0.7143       578
    satellite_pass     0.6267    0.5069    0.5605       361
scintillating_star     0.2438    0.2413    0.2425       286

          accuracy                         0.6525      3600
         macro avg     0.5916    0.5684    0.5722      3600
      weighted avg     0.6396    0.6525    0.6390      3600



In [13]:


# Combine the original features with the bandpass features.
F_e1_train = pd.concat(
    [F_train.reset_index(drop=True), F_bp_train.reset_index(drop=True)],
    axis=1
)

F_e1_test = pd.concat(
    [F_test.reset_index(drop=True), F_bp_test.reset_index(drop=True)],
    axis=1
)

# Ensure the test columns match the training columns exactly.
F_e1_test = F_e1_test.reindex(columns=F_e1_train.columns)

# Train on all labeled data using the best E1 setting.
model_e1_final = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.07,
    class_weight="balanced",
    random_state=0
)

model_e1_final.fit(F_e1_train, train["label"])

# Generate predictions.
pred_test_e1 = model_e1_final.predict(F_e1_test)

sub_e1 = pd.DataFrame({
    "id": test["id"],
    "label": pred_test_e1
})

print("Predicted label distribution:")
print(sub_e1["label"].value_counts())

sub_e1.to_csv(
    "../outputs/submissions/submission_e1.csv",
    index=False
)

print("Saved E1 submission.")

Predicted label distribution:
label
radar_chirp         6999
rfi_persistent       986
satellite_pass       743
pulsar               481
rfi_intermittent     426
frb                  350
eti_candidate         15
Name: count, dtype: int64
Saved E1 submission.


In [14]:
print("E1 prediction distribution:")
print(sub_e1["label"].value_counts())

print("\nE1 prediction proportions:")
print(sub_e1["label"].value_counts(normalize=True).round(3))

print("\nSubmission shape:", sub_e1.shape)
print("Unique IDs:", sub_e1["id"].nunique())

E1 prediction distribution:
label
radar_chirp         6999
rfi_persistent       986
satellite_pass       743
pulsar               481
rfi_intermittent     426
frb                  350
eti_candidate         15
Name: count, dtype: int64

E1 prediction proportions:
label
radar_chirp         0.700
rfi_persistent      0.099
satellite_pass      0.074
pulsar              0.048
rfi_intermittent    0.043
frb                 0.035
eti_candidate       0.002
Name: proportion, dtype: float64

Submission shape: (10000, 2)
Unique IDs: 10000


In [15]:

from scipy.stats import ks_2samp

# Compare only numeric features shared by train and test.
common = F_e1_train.columns.intersection(F_e1_test.columns)

rows = []

for col in common:
    a = F_e1_train[col].replace([np.inf, -np.inf], np.nan).dropna()
    b = F_e1_test[col].replace([np.inf, -np.inf], np.nan).dropna()

    if len(a) == 0 or len(b) == 0:
        continue

    pooled_std = np.sqrt((a.var() + b.var()) / 2)

    rows.append({
        "feature": col,
        "train_median": a.median(),
        "test_median": b.median(),
        "standardized_shift": (
            (b.mean() - a.mean()) / pooled_std
            if pooled_std > 1e-12 else 0.0
        ),
        "ks_statistic": ks_2samp(a, b).statistic
    })

shift = pd.DataFrame(rows)
shift["abs_shift"] = shift["standardized_shift"].abs()

print("Features with largest distribution differences:")
display(
    shift.sort_values("ks_statistic", ascending=False)
         .head(20)
         .round(3)
)

Features with largest distribution differences:


,feature,train_median,test_median,standardized_shift,ks_statistic,abs_shift
35,tsamp_s,18.250,16.000,0.000,1.000,0.000
36,channel_width_hz,2.790,3.810,0.000,1.000,0.000
49,bp_p3_std,13.156,17.465,3.747,0.958,3.747
57,bp_p5_std,13.150,17.470,3.784,0.958,3.784
41,bp_p1_std,13.167,17.478,3.704,0.955,3.704
25,p5_std,13.706,19.160,3.568,0.950,3.568
15,p3_std,13.720,19.159,3.535,0.950,3.535
5,p1_std,13.741,19.175,3.469,0.944,3.469
37,bp_p0_std,13.244,17.515,3.065,0.917,3.065
45,bp_p2_std,13.245,17.519,3.047,0.914,3.047


In [16]:
from pathlib import Path

FEATURE_DIR = Path("../outputs/features")
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

F_e1_train.to_pickle(FEATURE_DIR / "F_e1_train.pkl")
F_e1_test.to_pickle(FEATURE_DIR / "F_e1_test.pkl")

print("Saved E1 features:")
print(F_e1_train.shape, F_e1_test.shape)

Saved E1 features:
(18000, 65) (10000, 65)
